<a href="https://colab.research.google.com/github/BuildWithAdi25/House-Price-EDA-Project/blob/main/House_Price_EDA_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis of House Prices
**Dataset:** Ames Housing — Kaggle *House Prices: Advanced Regression Techniques* (`train.csv`)

## Objective
Explore the factors associated with house sale prices using Python, Pandas, NumPy, Matplotlib, Seaborn and SciPy. This is an **EDA-only project**; no machine-learning model is built.

## Guiding questions
1. How is `SalePrice` distributed? Is it skewed?
2. Which numerical features correlate most strongly with `SalePrice`?
3. Does `OverallQual` change price in an ordered way?
4. Do some neighborhoods consistently cost more?
5. Are there outliers, and are they errors or unusual sales?
6. Which missing values mean a feature is absent rather than unknown?

**Dataset source:** Kaggle — House Prices: Advanced Regression Techniques. Download `train.csv` from the competition Data page. In Colab, upload the file when prompted by the next cell.

**Notebook workflow:** load → understand → clean → univariate analysis → relationships → heatmap → outliers → dashboard → conclusions.


## 1. Setup and data loading
Import the required libraries and load `train.csv`. The notebook will ask you to upload the CSV if it is not already in the working directory.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 100)

# If train.csv is not present, upload it (works in Google Colab).
if not os.path.exists("train.csv"):
    try:
        from google.colab import files
        uploaded = files.upload()
        if "train.csv" not in uploaded:
            raise FileNotFoundError("Please upload the Kaggle file named train.csv.")
    except ImportError:
        raise FileNotFoundError("Place train.csv in the same folder as this notebook.")

df = pd.read_csv("train.csv")
print("Dataset shape:", df.shape)
display(df.head())
df.info()

missing = df.isna().sum().sort_values(ascending=False)
missing_summary = pd.DataFrame({
    "missing_count": missing[missing > 0],
    "missing_percent": (missing[missing > 0] / len(df) * 100).round(2)
})
display(missing_summary)


**What to note:** The expected dataset has about 1,460 rows and 81 columns. Record the actual shape and how many columns contain missing values. The largest missing-value counts often occur in features that many houses simply do not have (for example, a pool or alley access).

## 2. Classify variables
Separate numerical and categorical columns, identify ordinal quality columns, and create a compact variable summary.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

ordinal_orders = {
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"]
}
for col, order in ordinal_orders.items():
    if col in df.columns:
        df[col] = pd.Categorical(df[col], categories=order, ordered=True)

variable_summary = pd.DataFrame({
    "column": df.columns,
    "type": [
        "Numerical" if c in numeric_cols else
        "Categorical" if c in categorical_cols else "Other"
        for c in df.columns
    ],
    "unique_values": [df[c].nunique(dropna=True) for c in df.columns],
    "examples": [", ".join(map(str, df[c].dropna().unique()[:4])) for c in df.columns]
})
display(variable_summary)
print("Numerical columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Note: OverallQual is numeric in storage, but represents an ordinal quality rating.")


**What to note:** Numerical variables measure amounts or counts; categorical variables describe groups. `OverallQual` is stored as a number but its values represent ordered quality levels.

## 3. Descriptive statistics
Review the numerical summary and calculate centre and spread for `SalePrice`, `GrLivArea`, and `LotArea`.

In [ ]:
display(df[numeric_cols].describe().T)

for col in ["SalePrice", "GrLivArea", "LotArea"]:
    s = df[col].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    print(f"\\n{col}")
    print({
        "mean": round(s.mean(), 2),
        "median": round(s.median(), 2),
        "mode": s.mode().iloc[0],
        "std_dev": round(s.std(), 2),
        "variance": round(s.var(), 2),
        "range": round(s.max() - s.min(), 2),
        "IQR": round(q3 - q1, 2),
        "skewness": round(s.skew(), 3),
        "kurtosis": round(s.kurt(), 3)
    })

for col in ["Neighborhood", "HouseStyle"]:
    print(f"\\nValue counts for {col}:")
    display(df[col].value_counts().head(10))
    print("Mode:", df[col].mode().tolist())


**What to note:** Compare mean and median for `SalePrice`. If the mean is greater than the median, a smaller number of expensive houses may be pulling the average upward (right skew). Standard deviation and IQR describe spread.

## 4. Distributions
Plot sale price, its log transform, quality-group prices, neighborhood prices, and category counts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.histplot(df["SalePrice"], kde=True, ax=axes[0])
axes[0].set_title("SalePrice distribution")
sns.boxplot(x=df["SalePrice"], ax=axes[1])
axes[1].set_title("SalePrice boxplot")
sns.histplot(np.log1p(df["SalePrice"]), kde=True, ax=axes[2])
axes[2].set_title("Log-transformed SalePrice")
for ax in axes:
    ax.set_xlabel("Sale price (USD)" if ax is not axes[2] else "log(1 + SalePrice)")
plt.tight_layout()
plt.show()

print("SalePrice skewness:", round(df["SalePrice"].skew(), 3))
print("Log SalePrice skewness:", round(np.log1p(df["SalePrice"]).skew(), 3))

plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x="OverallQual", y="SalePrice")
plt.title("Sale price by overall quality")
plt.xlabel("Overall quality")
plt.ylabel("Sale price (USD)")
plt.show()

neighborhood_order = df.groupby("Neighborhood", observed=False)["SalePrice"].median().sort_values().index
plt.figure(figsize=(13, 6))
sns.boxplot(data=df, x="Neighborhood", y="SalePrice", order=neighborhood_order)
plt.xticks(rotation=90)
plt.title("Sale price by neighborhood (ordered by median)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=df, x="OverallQual", ax=axes[0])
axes[0].set_title("Count of houses by overall quality")
sns.countplot(data=df, x="HouseStyle", ax=axes[1])
axes[1].set_title("Count by house style")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


**What to note:** Check whether the log transformation reduces skewness. Look for an upward price pattern across `OverallQual` and differences between neighborhood medians. The plots show distributions, not proof that a feature causes a higher price.

## 5. Correlation and covariance
Calculate Pearson correlation and covariance for numerical variables, then inspect the ten strongest correlations with sale price.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
corr_matrix = numeric_df.corr(method="pearson")
cov_matrix = numeric_df.cov()

top_corr = corr_matrix["SalePrice"].drop("SalePrice").abs().sort_values(ascending=False).head(10)
top_corr_table = pd.DataFrame({
    "correlation_with_SalePrice": corr_matrix.loc[top_corr.index, "SalePrice"],
    "covariance_with_SalePrice": cov_matrix.loc[top_corr.index, "SalePrice"]
}).sort_values("correlation_with_SalePrice", key=abs, ascending=False)
display(top_corr_table)

print("Correlation is easier to compare because it is standardised between -1 and +1; covariance depends on the units and scale of the variables.")

if {"GarageArea", "GarageCars"}.issubset(corr_matrix.columns):
    print("GarageArea vs GarageCars correlation:", round(corr_matrix.loc["GarageArea", "GarageCars"], 3))


**What to note:** Correlation measures the strength and direction of a linear relationship. It does not prove causation. Correlation is easier to compare across differently scaled variables than covariance.

## 6. Scatter plots and pair plot
Visualise the relationships between sale price and living area, basement area, and year built.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
pairs = [("GrLivArea", "SalePrice"), ("TotalBsmtSF", "SalePrice"), ("YearBuilt", "SalePrice")]
for ax, (x, y) in zip(axes, pairs):
    sns.scatterplot(data=df, x=x, y=y, alpha=0.6, ax=ax)
    ax.set_title(f"{x} vs {y}")
plt.tight_layout()
plt.show()

unusual = df[(df["GrLivArea"] > 4000) & (df["SalePrice"] < 200000)]
print("Large living area with relatively low sale price (inspect these rows):", len(unusual))
display(unusual[["Id", "GrLivArea", "SalePrice", "OverallQual", "Neighborhood"]])

pair_cols = ["SalePrice", "GrLivArea", "TotalBsmtSF", "GarageArea", "YearBuilt"]
sns.pairplot(df[pair_cols].dropna().sample(min(500, df[pair_cols].dropna().shape[0]), random_state=42),
             diag_kind="kde", corner=True)
plt.show()


**What to note:** Scatter plots reveal trends and unusual observations. The pair plot gives a compact view of relationships among the selected numerical variables. The highlighted large-area/low-price rows deserve inspection rather than automatic deletion.

## 7. Correlation heatmap
Display a heatmap for the ten features most strongly correlated with `SalePrice`.

In [ ]:
top_features = ["SalePrice"] + top_corr.index.tolist()[:9]
corr_top = numeric_df[top_features].corr()
mask = np.triu(np.ones_like(corr_top, dtype=bool), k=1)

plt.figure(figsize=(11, 8))
sns.heatmap(corr_top, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5)
plt.title("Top 10 features correlated with SalePrice")
plt.tight_layout()
plt.show()


**What to note:** Strong positive values indicate that two variables tend to increase together; negative values indicate an inverse relationship. The masked half avoids displaying the same pair twice.

## 8. Outlier detection
Use both the IQR rule and z-scores for `GrLivArea` and `SalePrice`. Compare their counts and overlap.

In [ ]:
def iqr_outlier_mask(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

def zscore_outlier_mask(series, threshold=3):
    z = (series - series.mean()) / series.std()
    return z.abs() > threshold

outlier_masks = {}
for col in ["GrLivArea", "SalePrice"]:
    valid = df[col].notna()
    iqr_mask = iqr_outlier_mask(df.loc[valid, col])
    z_mask = zscore_outlier_mask(df.loc[valid, col])
    outlier_masks[col] = {
        "IQR": set(df.loc[valid].index[iqr_mask]),
        "Z-score": set(df.loc[valid].index[z_mask])
    }
    print(f"\\n{col}")
    print("IQR outliers:", len(outlier_masks[col]["IQR"]))
    print("Z-score outliers:", len(outlier_masks[col]["Z-score"]))
    print("Overlap:", len(outlier_masks[col]["IQR"] & outlier_masks[col]["Z-score"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(y=df["GrLivArea"], ax=axes[0])
axes[0].set_title("GrLivArea outliers")
sns.boxplot(y=df["SalePrice"], ax=axes[1])
axes[1].set_title("SalePrice outliers")
plt.tight_layout()
plt.show()

iqr_union = outlier_masks["GrLivArea"]["IQR"] | outlier_masks["SalePrice"]["IQR"]
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="GrLivArea", y="SalePrice", alpha=0.45)
sns.scatterplot(data=df.loc[list(iqr_union)], x="GrLivArea", y="SalePrice",
                color="red", label="IQR flagged")
plt.title("GrLivArea vs SalePrice with IQR-flagged observations")
plt.legend()
plt.show()

display(df.loc[list(iqr_union), ["Id", "GrLivArea", "SalePrice", "OverallQual"]].head(20))


**What to note:** IQR uses quartiles and is less affected by extreme values. Z-scores use mean and standard deviation, so skewed distributions can make their outlier flags behave differently. A flagged point is not automatically an error; inspect it before removing it.

## 9. Data quality checks and cleaning
Check duplicates, standardise categorical text, handle missing values with dataset-aware decisions, and check impossible values. Keep the original `df` unchanged by creating `df_clean`.

In [ ]:
df_clean = df.copy()

# Duplicate rows
duplicate_count = df_clean.duplicated().sum()
print("Duplicate rows:", duplicate_count)
df_clean = df_clean.drop_duplicates()
print("Shape after removing duplicates:", df_clean.shape)

# Standardise whitespace and casing for text categories.
text_cols = df_clean.select_dtypes(include="object").columns
for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

# In these Ames columns, NA commonly means the feature is absent.
absent_means_none = ["PoolQC", "Alley", "Fence", "FireplaceQu", "GarageType",
                     "GarageFinish", "GarageQual", "GarageCond", "BsmtQual",
                     "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
                     "MasVnrType"]
for col in absent_means_none:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("None")

# Fill remaining categorical missing values with the most frequent category.
for col in df_clean.select_dtypes(include=["object", "category"]).columns:
    if df_clean[col].isna().any():
        mode_value = df_clean[col].mode().iloc[0]
        if isinstance(df_clean[col].dtype, pd.CategoricalDtype) and mode_value not in df_clean[col].cat.categories:
            df_clean[col] = df_clean[col].cat.add_categories([mode_value])
        df_clean[col] = df_clean[col].fillna(mode_value)

# Fill remaining numerical missing values with the column median.
for col in df_clean.select_dtypes(include=np.number).columns:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

remaining_missing = df_clean.isna().sum().sum()
print("Remaining missing values:", remaining_missing)
print("Cleaned shape:", df_clean.shape)

# Check impossible values
if {"YearBuilt", "YrSold"}.issubset(df_clean.columns):
    print("Rows where YearBuilt > YrSold:", (df_clean["YearBuilt"] > df_clean["YrSold"]).sum())
for col in ["GrLivArea", "LotArea", "TotalBsmtSF", "GarageArea"]:
    if col in df_clean.columns:
        print(f"Negative values in {col}:", (df_clean[col] < 0).sum())

print("\\nCleaning decisions: NA in feature-presence columns was treated as 'None'; other categorical values used the mode; remaining numeric values used the median.")


**What to note:** Missing values do not always mean unknown information. For features such as pools, fences, and garages, a blank can mean the house does not have that feature. Median/mode filling is a simple EDA preparation choice; document it and do not treat it as a universal rule.

## 10. Final dashboard
Combine four useful views into one 2×2 dashboard and save it as `eda_dashboard.png`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: average sale price by year built
year_price = df_clean.groupby("YearBuilt")["SalePrice"].mean().sort_index()
axes[0, 0].plot(year_price.index, year_price.values, label="Average price")
axes[0, 0].set_title("Average sale price by year built")
axes[0, 0].set_xlabel("Year built")
axes[0, 0].set_ylabel("Average sale price (USD)")
axes[0, 0].legend()

# Panel 2: top 10 neighborhoods by mean sale price
neighborhood_price = df_clean.groupby("Neighborhood")["SalePrice"].mean().nlargest(10).sort_values()
neighborhood_price.plot(kind="barh", ax=axes[0, 1])
axes[0, 1].set_title("Top 10 neighborhoods by average price")
axes[0, 1].set_xlabel("Average sale price (USD)")
axes[0, 1].set_ylabel("Neighborhood")

# Panel 3: sale price distribution with mean and median
sns.histplot(df_clean["SalePrice"], kde=True, ax=axes[1, 0])
axes[1, 0].axvline(df_clean["SalePrice"].mean(), linestyle="--",
                   label="Mean")
axes[1, 0].axvline(df_clean["SalePrice"].median(), linestyle=":",
                   label="Median")
axes[1, 0].set_title("Sale price distribution")
axes[1, 0].set_xlabel("Sale price (USD)")
axes[1, 0].legend()

# Panel 4: living area vs price coloured by quality
sns.scatterplot(data=df_clean, x="GrLivArea", y="SalePrice",
                hue="OverallQual", palette="viridis", alpha=0.7,
                ax=axes[1, 1], legend=False)
axes[1, 1].set_title("Living area vs sale price by quality")
axes[1, 1].set_xlabel("Above-ground living area (sq ft)")
axes[1, 1].set_ylabel("Sale price (USD)")

fig.suptitle("House Price EDA Summary", fontsize=16)
plt.tight_layout()
plt.savefig("eda_dashboard.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved dashboard as eda_dashboard.png")


**What to note:** The dashboard summarises year-built trends, expensive neighborhoods, the sale-price distribution, and the relationship between living area, price, and quality. The image is saved in the notebook's working directory.

## 11. Conclusions
Complete these points after running the notebook. Use your actual outputs rather than assuming the expected findings.

1. **Sale price distribution:** The histogram and skewness values show whether `SalePrice` is right- or left-skewed. Compare its skewness with the log-transformed values.
2. **Strong relationships:** The correlation table identifies the numerical features most strongly associated with `SalePrice`.
3. **Overall quality:** The boxplot shows how sale-price medians and spread change across quality ratings.
4. **Neighborhood:** The neighborhood boxplot and dashboard identify areas with higher typical sale prices.
5. **Outliers:** IQR and z-score counts differ because the methods use different definitions. Flagged houses were inspected rather than automatically deleted.
6. **Missing values:** Some missing entries represent an absent feature (such as no pool or fence), not unknown data.

### Limitations
This is an exploratory analysis of one dataset. Correlation does not establish causation, and cleaning choices can affect results. No machine-learning model or price prediction is included.
